# 09 · Test of H4: do demonstrations that show corrections close the gap?

**Hypothesis H4** (README): scripted rollouts follow their plan almost perfectly, so the data never shows how to get back on track after a small error. The policy's own small errors then take it to states it has never seen, and grasps and placements fail. Teleoperated demos contain such corrections naturally.

**Intervention (DART, Laskey et al. 2017):** during generation, the executed motion is disturbed by small, slowly drifting noise (`--action-noise 0.05`, correlation time 5 steps = 0.25 s), while the recorded action is always the teacher's clean action, computed from the disturbed pose: the data now contains off-path states labelled with the correction back to the plan. Calibrated in physics-only rollouts: the gripper deviates from the clean path by a median 8 mm (90th percentile 15 mm), the teacher's yield drops from ~85% to ~75-80%.

**Design.** Two runs of this notebook, in two runtimes in parallel, differing ONLY in `VARIANT`:
* `clean`: the 5 consistent demos of notebook 08, 130 fresh scenes each, no noise.
* `dart`: the same 650 scenes and plans (same seeds), with noise.
Both datasets are cut to the same 450 successes, spread evenly over the 5 demos, and trained and evaluated exactly like every other run. `clean` vs notebook 08 (~250 episodes, 59%) measures data size; `dart` vs `clean` measures corrective data.

**Predictions, fixed before running** (success pooled over 2 evaluations, n = 100 each):
* H4 confirmed: `dart` >= `clean` + 15 points, with fewer missed grasps, and temporal ensembling no longer collapses.
* H4 rejected: `dart` <= `clean` + 5 points.
* In between: suggestive only.
* Data size: if `clean` >= 70%, the placement failures of notebook 08 were a data-size effect.

**Runtime:** T4, about 3 h per variant (generation ~75 min, datasets ~10 min, training ~75 min, three evaluations ~30 min).

In [ ]:
import os, sys, json, subprocess
os.environ["MUJOCO_GL"] = "egl"                      # GPU rendering, must be set before mujoco is imported
REPO_URL = "https://github.com/LukasWill/phone2panda.git"
BASE = "/content" if os.path.exists("/content") else "/kaggle/working"   # Colab or Kaggle
CODE_DIR = f"{BASE}/phone2panda"
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "-q"], check=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lerobot[libero]==0.6.1", "imageio-ffmpeg"],
                   capture_output=True, text=True)
if r.returncode != 0:                                  # show pip's own reason, not just "exit status 1"
    print((r.stdout + r.stderr)[-4000:]); raise RuntimeError("pip install failed: see the lines above")
# Colab preinstalls an older torchao; a newer diffusers (pulled in by lerobot) then fails on import
# (FqnToConfig). ACT does not use torchao, and diffusers skips it when it is absent.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
from p2p.sim.env import ensure_nvidia_egl
ensure_nvidia_egl()      # Colab: make EGL render on the GPU (otherwise it silently renders on the CPU, ~10x slower)
# Hugging Face token (Colab: key icon -> Secrets -> HF_TOKEN; Kaggle: Add-ons -> Secrets -> HF_TOKEN)
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN found: uploads to the Hub will fail")
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import whoami
    print("Hugging Face account:", whoami(token=os.environ["HF_TOKEN"])["name"])
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
VARIANT = "clean"          # "clean" or "dart": run this notebook twice, once with each (two tabs / runtimes)
NOISE = {"clean": 0.0, "dart": 0.05}[VARIANT]
DEMOS = "f9115215f9_lefthand,c5f525e833,9a63cb1538_lefthand,92e0b86bc6_lefthand,65ee6ec786"   # grasp sides 106-116°
PER_DEMO, N_EPISODES, SEED0 = 130, 450, 2_000_000     # same seeds for both variants: paired scenes and plans
HUB_REPO, SEED, STEPS, BATCH = "phone2panda-data", 0, 20000, 32
NAME = f"ours_consistent_{VARIANT}"
RUN = f"act_{NAME}_s{SEED}"
GEN = f"{BASE}/gen_{VARIANT}"
WORKERS = os.cpu_count()
from p2p.sim.env import render_speed
r = render_speed(); print(r)
assert r["gpu_rendering"], "rendering is on the CPU: Runtime -> Change runtime type -> T4 GPU, then restart"

## Generate (resumable: rerun the cell after a disconnect and it continues)

In [ ]:
!python -m p2p.data.generate data/processed {GEN} --demos {DEMOS} --per-demo {PER_DEMO} --seed0 {SEED0} --action-noise {NOISE} --workers {WORKERS}

In [ ]:
import pandas as pd
log = pd.read_json(f"{GEN}/generation_log.jsonl", lines=True).drop_duplicates("seed", keep="last")
print(VARIANT, "yield", round(log.success.mean(), 3), "successes", int(log.success.sum()), "of", len(log))
display(log.groupby("demo").success.agg(["mean", "sum"]))
assert log.success.sum() >= N_EPISODES, "fewer successes than N_EPISODES: lower N_EPISODES in BOTH variants"

## Dataset: the same number of successes for both variants, spread evenly over the demos

In [ ]:
!python -m p2p.data.lerobot_io ours {GEN} {BASE}/lerobot/{NAME} --balanced {N_EPISODES}
!python tools/demo_consistency.py {BASE}/lerobot/{NAME}
from p2p.hub import push
push(f"{BASE}/lerobot/{NAME}", HUB_REPO, path_in_repo=f"lerobot/{NAME}")

## Train and evaluate (same settings as every other run)

In [ ]:
!python -m p2p.policy.act train {BASE}/lerobot/{NAME} {BASE}/runs/{RUN} --steps {STEPS} --batch-size {BATCH} --seed {SEED}

In [ ]:
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 3 --out {BASE}/runs/{RUN}/eval
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 0 --out {BASE}/runs/{RUN}/eval_rerun
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 0 --temporal-ensemble 0.01 --out {BASE}/runs/{RUN}/eval_te
import glob
evals = " ".join(sorted(glob.glob(f"{BASE}/runs/{RUN}/eval*/eval.json")))
!python tools/policy_failures.py {evals}

In [ ]:
from p2p.hub import push
push(f"{BASE}/runs/{RUN}", HUB_REPO, path_in_repo=f"runs/{RUN}")
!cp {GEN}/generation_log.jsonl {BASE}/runs/{RUN}/generation_log.jsonl
!cd {BASE}/runs && zip -qr {BASE}/{RUN}.zip {RUN} -x "*/final/*" "*/step_*"
try:
    from google.colab import files; files.download(f"{BASE}/{RUN}.zip")
except Exception:
    print("download", f"{BASE}/{RUN}.zip", "from the Output panel")